# Team 04 End-to-End LangGraph Node Notebook

This notebook is the prompt-driven end-to-end harness for the active Team 04 LangGraph runtime.

It is meant to test one agent-node loop from user prompt through tool calls to a final state, while keeping site context notebook-local and inspectable.

## Scope

Use this notebook when you want to test a LangGraph agent-node style loop against real prompts:
1. start from a user prompt and notebook-provided site state
2. let the active planner and supervisor choose the next action
3. run the local tool surface until the loop reaches a final state

This notebook now includes:
- prompt scenarios for one-building and two-building runs
- notebook-local site objects such as streets or alignment guides
- **Phase 0 comprehension surfaced from the run** — the typed `design_brief` and the canonical `site_model` the agent now builds at graph start (`START → extract_brief → planner`)
- visualization of the current candidate, placed buildings, and graph payload

This is an **in-process** end-to-end run via `run_agent` (the same graph the FastAPI backend drives). The backend's `extract_brief` node also streams a `brief` decision node over SSE, which the `frontend/` decision graph renders — this notebook is the offline equivalent for inspecting the same `design_brief`/`site_model` directly.

Keep `tool_dev_mode.ipynb` for deterministic geometry debugging. Use this notebook for prompt-to-final-state behavior.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

workspace_root = Path.cwd().resolve()
candidate_roots = (
    workspace_root,
    workspace_root.parent,
    workspace_root / "team_04",
    workspace_root.parent / "team_04",
)
TEAM_ROOT = next((path for path in candidate_roots if (path / "agent").exists()), None)
if TEAM_ROOT is None:
    raise FileNotFoundError(
        "Run this notebook from the workspace root, the team_04 folder, or the team_04/test_notebooks folder."
    )

team_root_str = str(TEAM_ROOT)
if team_root_str not in sys.path:
    sys.path.insert(0, team_root_str)

E2E_OUTPUT_PATH = TEAM_ROOT / "test_notebooks" / "end_to_end_api_agent_output.json"
E2E_OUTPUT_PATH

WindowsPath('C:/Users/tuemi/Downloads/Glabtools/IAAC Repo/bimsc26-datamgmt-session03/AIA26_Studio/team_04/test_notebooks/end_to_end_api_agent_output.json')

In [2]:
from langchain_openai import ChatOpenAI

import plotly.graph_objects as go

from agent.config import load_settings
from agent.decision_engine import OpenAIDecisionEngine, RuleBasedPlanner
from agent.graph import run_agent
from agent.mcp_client import CompositeToolClient, build_default_local_tool_client
from agent.notebook_demo_tools import build_notebook_demo_tool_client
from agent.tool_catalog import ToolCatalog

settings_error = None
try:
    settings = load_settings()
except Exception as exc:
    settings = None
    settings_error = str(exc)

{
    "llm_provider": settings.llm_provider if settings is not None else None,
    "llm_model": settings.llm_model if settings is not None else None,
    "settings_error": settings_error,
}

c:\Users\tuemi\miniconda3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


{'llm_provider': 'openai', 'llm_model': 'gpt-5-nano', 'settings_error': None}

In [5]:
def _to_xyz(point):
    if len(point) >= 3:
        return (float(point[0]), float(point[1]), float(point[2]))
    return (float(point[0]), float(point[1]), 0.0)


def _dedupe_boundary(boundary):
    cleaned = []
    for point in boundary or []:
        xyz = _to_xyz(point)
        if not cleaned or any(abs(cleaned[-1][index] - xyz[index]) > 1e-6 for index in range(3)):
            cleaned.append(xyz)
    if len(cleaned) > 1 and all(abs(cleaned[0][index] - cleaned[-1][index]) <= 1e-6 for index in range(3)):
        cleaned = cleaned[:-1]
    return cleaned


def _normalize_points(points):
    normalized = []
    for point in points or []:
        if isinstance(point, (list, tuple)) and point and isinstance(point[0], (list, tuple)):
            normalized.extend(_normalize_points(point))
            continue
        normalized.append(_to_xyz(point))
    return normalized


def _apply_zoom_to_fit(figure):
    figure.update_yaxes(scaleanchor="x", scaleratio=1, visible=False)
    figure.update_xaxes(visible=False)
    figure.update_layout(
        margin=dict(l=0, r=0, t=48, b=0),
        plot_bgcolor="#f8fafc",
        paper_bgcolor="#f8fafc",
        font=dict(color="#111827", size=13),
    )
    return figure


def _add_boundary_trace(figure, boundary, *, line_color, line_width, fillcolor=None):
    points = _dedupe_boundary(boundary)
    if len(points) < 2:
        return
    xs = [point[0] for point in points] + [points[0][0]]
    ys = [point[1] for point in points] + [points[0][1]]
    figure.add_trace(
        go.Scatter(
            x=xs,
            y=ys,
            mode="lines",
            line=dict(color=line_color, width=line_width),
            fill="toself" if fillcolor else None,
            fillcolor=fillcolor,
            hoverinfo="skip",
            showlegend=False,
        )
    )


def _add_site_object_traces(figure, site_objects):
    palette = {
        "street": "#2563eb",
        "street_centerline": "#2563eb",
        "alignment_guide": "#7c3aed",
        "setback_edge": "#0f766e",
    }
    for item in site_objects or []:
        if not isinstance(item, dict):
            continue
        points = _normalize_points(item.get("points", []))
        if len(points) < 2:
            continue
        color = palette.get(str(item.get("type", "")).strip(), "#64748b")
        label = str(item.get("name", item.get("type", "site_object"))).strip() or "site_object"
        xs = [point[0] for point in points]
        ys = [point[1] for point in points]
        figure.add_trace(
            go.Scatter(
                x=xs,
                y=ys,
                mode="lines+text",
                line=dict(color=color, width=3, dash="dash"),
                text=[label] + [""] * (len(xs) - 1),
                textposition="top center",
                textfont=dict(color=color, size=12),
                hoverinfo="skip",
                showlegend=False,
            )
        )


def _add_centerline_graph_trace(figure, building_graph):
    centerline_graph = (building_graph or {}).get("centerline_graph", {})
    nodes = centerline_graph.get("nodes", [])
    edges = centerline_graph.get("edges", [])
    if not nodes or not edges:
        return

    for edge in edges:
        start = nodes[edge["from_node_index"]]["point"]
        end = nodes[edge["to_node_index"]]["point"]
        figure.add_trace(
            go.Scatter(
                x=[start[0], end[0]],
                y=[start[1], end[1]],
                mode="lines",
                line=dict(color="#dc2626", width=4),
                hoverinfo="skip",
                showlegend=False,
            )
        )

    figure.add_trace(
        go.Scatter(
            x=[node["point"][0] for node in nodes],
            y=[node["point"][1] for node in nodes],
            mode="markers+text",
            marker=dict(color="#111827", size=9),
            text=[f"n{node['node_index']}" for node in nodes],
            textposition="top center",
            textfont=dict(color="#111827", size=11),
            hoverinfo="skip",
            showlegend=False,
        )
    )


def extract_latest_shape(final_state):
    shape_context = final_state.get("shape_context") or {}
    if isinstance(shape_context, dict) and isinstance(shape_context.get("boundary"), list):
        return shape_context

    for record in reversed(final_state.get("tool_history", [])):
        if not isinstance(record, dict):
            continue
        output = record.get("output") or {}
        data = output.get("data") if isinstance(output, dict) else None
        if isinstance(data, dict) and isinstance(data.get("boundary"), list):
            return data

    raise ValueError("No generated boundary payload was found in final_state.")


def make_decision_trace_figure(messages):
    rows = [
        message
        for message in messages
        if message.startswith(("Planner updated", "Supervisor decision", "Tool ", "Final report"))
    ]
    figure = go.Figure(
        data=[
            go.Table(
                header=dict(values=["Step", "Trace message"], fill_color="#dbeafe", align="left"),
                cells=dict(
                    values=[list(range(1, len(rows) + 1)), rows],
                    fill_color="#f8fafc",
                    align="left",
                ),
            )
        ]
    )
    figure.update_layout(title="Planner and supervisor trace")
    return figure


def make_building_graph_figure(shape_payload, *, title):
    figure = go.Figure()
    _add_centerline_graph_trace(figure, shape_payload.get("building_graph", {}))
    if not figure.data:
        raise ValueError("No drawable centerline graph was found in the supplied shape payload.")
    figure.update_layout(title=title)
    return _apply_zoom_to_fit(figure)


def make_site_plan_figure(site_boundary, *, current_shape=None, placed_buildings=None, site_objects=None, title):
    figure = go.Figure()
    _add_boundary_trace(figure, site_boundary, line_color="#1d4ed8", line_width=4)
    _add_site_object_traces(figure, site_objects)

    placed_palette = [
        ("#0f766e", "rgba(15, 118, 110, 0.18)"),
        ("#7c3aed", "rgba(124, 58, 237, 0.18)"),
        ("#c2410c", "rgba(194, 65, 12, 0.18)"),
    ]
    for index, item in enumerate(placed_buildings or []):
        if not isinstance(item, dict):
            continue
        boundary = item.get("boundary")
        if not isinstance(boundary, list):
            continue
        line_color, fill_color = placed_palette[index % len(placed_palette)]
        _add_boundary_trace(
            figure,
            boundary,
            line_color=line_color,
            line_width=3,
            fillcolor=fill_color,
        )

    if isinstance(current_shape, dict):
        _add_boundary_trace(
            figure,
            current_shape.get("boundary", []),
            line_color="#ea580c",
            line_width=3,
            fillcolor="rgba(249, 115, 22, 0.18)",
        )
        _add_centerline_graph_trace(figure, current_shape.get("building_graph", {}))

    figure.update_layout(title=title)
    return _apply_zoom_to_fit(figure)

In [6]:
SITE_BOUNDARY = [
    [0.0, 0.0, 0.0],
    [90.0, 0.0, 0.0],
    [90.0, 60.0, 0.0],
    [0.0, 60.0, 0.0],
    [0.0, 0.0, 0.0],
]

SITE_OBJECTS = [
    {
        "name": "Main Street",
        "type": "street",
        "points": [[0.0, 8.0, 0.0], [90.0, 8.0, 0.0]],
    },
    {
        "name": "East Service Street",
        "type": "street",
        "points": [[78.0, 0.0, 0.0], [78.0, 60.0, 0.0]],
    },
    {
        "name": "Central Alignment Guide",
        "type": "alignment_guide",
        "points": [[18.0, 0.0, 0.0], [18.0, 60.0, 0.0]],
    },
]

PROMPT_SCENARIOS = [
    {
        "name": "Single building near Main Street",
        "prompt": (
            "Place one U-shaped building of about 900 square meters inside the site boundary. "
            "Keep the wing graph readable and align the building roughly with Main Street if that helps."
        ),
        "layout_payload": {
            "workflow_mode": "full",
            "site_boundary": SITE_BOUNDARY,
            "site_objects": SITE_OBJECTS,
            "target_building_count": 1,
            "building_intents": [
                "Keep the wing graph readable for later wing-level edits and street alignment requests.",
            ],
        },
    },
    {
        "name": "Two buildings with street-aware placement",
        "prompt": (
            "Place two buildings inside the site boundary. "
            "Put the first building closer to Main Street and keep its graph readable for later edits. "
            "Then place a second building after the first one, keeping site access clear and using East Service Street or the Central Alignment Guide if helpful."
        ),
        "layout_payload": {
            "workflow_mode": "full",
            "site_boundary": SITE_BOUNDARY,
            "site_objects": SITE_OBJECTS,
            "target_building_count": 2,
            "requested_positions": [[24.0, 20.0], [62.0, 34.0]],
            "building_intents": [
                "Primary building close to Main Street with a clear wing graph.",
                "Second building should fit after the first placement and stay aware of street access.",
            ],
        },
    },
]

scenario_index = 1
selected_scenario = PROMPT_SCENARIOS[scenario_index]
prompt = selected_scenario["prompt"]
layout_payload = selected_scenario["layout_payload"]

{
    "selected_scenario": selected_scenario["name"],
    "prompt": prompt,
    "target_building_count": layout_payload["target_building_count"],
    "site_object_names": [item["name"] for item in SITE_OBJECTS],
    "settings_error": settings_error,
    "llm_provider": settings.llm_provider if settings is not None else None,
    "llm_model": settings.llm_model if settings is not None else None,
}

{'selected_scenario': 'Two buildings with street-aware placement',
 'prompt': 'Place two buildings inside the site boundary. Put the first building closer to Main Street and keep its graph readable for later edits. Then place a second building after the first one, keeping site access clear and using East Service Street or the Central Alignment Guide if helpful.',
 'target_building_count': 2,
 'site_object_names': ['Main Street',
  'East Service Street',
  'Central Alignment Guide'],
 'settings_error': None,
 'llm_provider': 'openai',
 'llm_model': 'gpt-5-nano'}

In [7]:
if settings is None:
    raise RuntimeError(
        "Missing LLM runtime settings. Define LLM_PROVIDER and provider credentials in the repo .env or team_04/.env before running the end-to-end agent cell."
    )

notebook_client = build_notebook_demo_tool_client(
    SITE_BOUNDARY,
    site_summary="Notebook-local site context for LangGraph node testing with site objects and optional second-building placement.",
    site_objects=SITE_OBJECTS,
    setback_m=5.0,
    spatial_intention_score=0.91,
    performance_score=0.88,
    shape_integrity_score=0.94,
 )
tool_client = CompositeToolClient([build_default_local_tool_client(), notebook_client])
catalog = ToolCatalog.from_discovered_tools(tool_client.list_tools())

llm = ChatOpenAI(
    api_key=settings.api_key,
    base_url=settings.base_url,
    model=settings.llm_model,
    timeout=settings.request_timeout_seconds,
    temperature=0,
 )
decision_engine = OpenAIDecisionEngine(
    llm=llm,
    decision_provider=settings.decision_llm_provider or settings.llm_provider,
    decision_model=settings.decision_llm_model or settings.llm_model,
    report_provider=settings.report_llm_provider or settings.llm_provider,
    report_model=settings.report_llm_model or settings.llm_model,
 )

final_state = run_agent(
    user_prompt=prompt,
    decision_engine=decision_engine,
    tool_client=tool_client,
    catalog=catalog,
    initial_layout=layout_payload,
    max_optimization_cycles=3,
    planner=RuleBasedPlanner(),
)

## Phase 0 — comprehension surfaced from the run

The agent now extracts a typed `DesignBrief` and builds a canonical `SiteModel` at the start of the
graph (`START → extract_brief → planner`). They land in `final_state["design_brief"]` and
`final_state["site_model"]` — the *same* data the backend streams as a `brief` decision node and the
`frontend/` decision graph renders. This cell surfaces them so the end-to-end run shows *what the
agent understood* before it placed anything.

In [8]:
# Phase 0 — what the agent comprehended before acting
design_brief = final_state.get("design_brief") or {}
site_model = final_state.get("site_model") or {}

print("=== DESIGN BRIEF (typed comprehension) ===")
if design_brief:
    print("source        :", design_brief.get("source"))
    print("building_count:", design_brief.get("building_count"))
    for i, b in enumerate(design_brief.get("buildings", [])):
        print(f"  building {i + 1}: shape={b.get('shape_preference')} "
              f"area={b.get('footprint_area_sqm')} storeys={b.get('storeys')} use={b.get('use')}")
    print("courtyard     :", design_brief.get("courtyard_requested"), design_brief.get("courtyard_qualities"))
    print("parking       :", design_brief.get("parking_requested"))
    print("weights       : view=%s sun=%s align=%s" % (
        design_brief.get("view_weight"), design_brief.get("sun_weight"), design_brief.get("alignment_weight")))
    print("ambiguities   :", design_brief.get("ambiguities") or "(none)")
else:
    print("(no design_brief in final_state — check the extract_brief node in agent/graph.py)")

print("\n=== SITE MODEL (canonical structured site) ===")
if site_model.get("available"):
    setbacks = site_model.get("setbacks") or {}
    print("corners       :", len(site_model.get("corners", [])))
    print("sides         :", len(site_model.get("sides", [])))
    print("site area     :", setbacks.get("site_area_sqm"))
    print("buildable area:", setbacks.get("buildable_area_sqm"))
    print("phase slots   : roads=%s grid=%s sun=%s" % (
        site_model.get("roads"), site_model.get("grid"), site_model.get("sun")))
else:
    print("site_model unavailable:", site_model.get("reason", "(no site_model key)"))

{"design_brief_source": design_brief.get("source"), "site_model_available": site_model.get("available")}

=== DESIGN BRIEF (typed comprehension) ===
source        : llm
building_count: 2
  building 1: shape=auto area=None storeys=None use=residential
  building 2: shape=auto area=None storeys=None use=residential
courtyard     : False []
parking       : False
weights       : view=0.5 sun=0.5 align=0.8
ambiguities   : ['Exact footprint area, number of storeys, and building use are not specified.', "Definition of 'after the first one' in site plan order is ambiguous (direction, offset, or alignment).", 'No explicit site boundary, access points, or driveway configuration provided.', 'No preferred orientation or rotation constraints beyond alignment cues.']

=== SITE MODEL (canonical structured site) ===
corners       : 4
sides         : 4
site area     : 5400.0
buildable area: 4000.0
phase slots   : roads=None grid=None sun=None


{'design_brief_source': 'llm', 'site_model_available': True}

In [9]:
decision_trace = [
    message
    for message in final_state.get("messages", [])
    if message.startswith(("Design brief", "Planner updated", "Supervisor decision", "Tool ", "Final report"))
]
placed_buildings = final_state.get("placed_buildings", [])
design_brief = final_state.get("design_brief") or {}
site_model = final_state.get("site_model") or {}

summary = {
    "selected_scenario": selected_scenario["name"],
    "prompt": prompt,
    "final_response": final_state.get("final_response"),
    "target_building_count": layout_payload.get("target_building_count"),
    "placed_building_count": len(placed_buildings),
    "placed_building_geometry_ids": [
        item.get("geometry_id")
        for item in placed_buildings
        if isinstance(item, dict)
    ],
    "site_object_names": [item["name"] for item in SITE_OBJECTS],
    "workflow_mode": final_state.get("workflow_mode"),
    "violations": final_state.get("violations"),
    "placement_fit_summary": final_state.get("placement_fit_summary"),
    # Phase 0 comprehension surfaced in the saved output
    "design_brief": design_brief,
    "site_model_available": site_model.get("available"),
    "tool_sequence": [
        record.get("tool")
        for record in final_state.get("tool_history", [])
        if isinstance(record, dict)
    ],
    "decision_trace": decision_trace,
}

E2E_OUTPUT_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary

{'selected_scenario': 'Two buildings with street-aware placement',
 'prompt': 'Place two buildings inside the site boundary. Put the first building closer to Main Street and keep its graph readable for later edits. Then place a second building after the first one, keeping site access clear and using East Service Street or the Central Alignment Guide if helpful.',
 'final_response': 'Workflow completed without any pending plan steps.',
 'target_building_count': 2,
 'placed_building_count': 1,
 'placed_building_geometry_ids': ['generate_building_boundary_1486ed9f1700'],
 'site_object_names': ['Main Street',
  'East Service Street',
  'Central Alignment Guide'],
 'workflow_mode': 'full',
 'violations': [],
 'placement_fit_summary': {},
 'design_brief': {'building_count': 2,
  'buildings': [{'shape_preference': 'auto',
    'footprint_area_sqm': None,
    'storeys': None,
    'use': 'residential',
    'intent_text': 'Place near Main Street; keep graph readable for later edits.'},
   {'shape

In [10]:
make_decision_trace_figure(final_state.get("messages", []))

In [11]:
latest_shape = extract_latest_shape(final_state)

# The centerline (wing) graph only exists for multi-wing footprints (U / L / T / H / Y / X).
# When the brief's shape is "auto" or a simple bar, there is no wing skeleton — so fall back to
# the boundary plot instead of raising, rather than assuming every shape has a centerline graph.
_centerline = (latest_shape or {}).get("building_graph", {}).get("centerline_graph", {})
if _centerline.get("nodes") and _centerline.get("edges"):
    figure = make_building_graph_figure(
        latest_shape,
        title="Latest candidate centerline graph after the LangGraph node run",
    )
else:
    print("No centerline/wing graph in the latest shape (this run's footprint has no wings) — "
          "showing the building boundary instead.")
    figure = make_site_plan_figure(
        SITE_BOUNDARY,
        current_shape=latest_shape,
        site_objects=SITE_OBJECTS,
        title="Latest candidate boundary (no centerline graph for this shape)",
    )
figure

No centerline/wing graph in the latest shape (this run's footprint has no wings) — showing the building boundary instead.


In [12]:
latest_shape = extract_latest_shape(final_state)

make_site_plan_figure(
    SITE_BOUNDARY,
    current_shape=latest_shape,
    placed_buildings=final_state.get("placed_buildings", []),
    site_objects=SITE_OBJECTS,
    title="Prompt-driven LangGraph node run with site objects and placed buildings",
)

In [13]:
{
    "plan": final_state.get("plan"),
    "placed_buildings": final_state.get("placed_buildings"),
    "site_context_keys": sorted((final_state.get("site_context") or {}).keys()),
    "saved_to": str(E2E_OUTPUT_PATH),
}

{'plan': [{'step_id': 'read_site',
   'action': 'read_site',
   'goal': 'Load site boundary, context, and legal constraints.',
   'status': 'completed'},
  {'step_id': 'generate_shape',
   'action': 'generate_shape',
   'goal': 'Create the next geometry candidate for building 2. Intent: Second building should fit after the first placement and stay aware of street access.',
   'status': 'skipped'},
  {'step_id': 'check_requested_position',
   'action': 'check_requested_position',
   'goal': "Check whether the user's requested position works for building 2. Intent: Second building should fit after the first placement and stay aware of street access.",
   'status': 'skipped'},
  {'step_id': 'check_constraints',
   'action': 'check_constraints',
   'goal': 'Validate the current geometry for building 2 against all constraints. Intent: Second building should fit after the first placement and stay aware of street access.',
   'status': 'skipped'},
  {'step_id': 'optimize',
   'action': 'optim

## Interactive clarification — the agent asks back

When a prompt is too vague to place accurately **and** `interactive_clarification` is enabled, the
agent now pauses at `await_human` and returns a **structured question**
(`final_state["clarification_request"]`) instead of guessing — building shape, preferred side, the
**view-optimisation side**, size, use, and count. Policy: *ask only on critical gaps* (shape /
side / view side); minor gaps fall back to documented defaults.

This is the exact payload the backend streams as a `clarify` decision node and the `frontend/`
`ClarifyPanel` renders as chips. Below: a vague prompt → the question → we answer it (simulating the
user clicking chips) → a second run places the building accurately. This exercises the **real LLM**
(`decision_engine`) the same way the API does.

In [14]:
# --- Run 1: a deliberately vague prompt with interactive clarification ON ---
from agent.clarify import apply_clarification_answers

VAGUE_PROMPT = "Place a building on the site."
vague_layout = {
    "workflow_mode": "full",
    "site_boundary": SITE_BOUNDARY,
    "site_objects": SITE_OBJECTS,
    "interactive_clarification": True,
}

clarify_state = run_agent(
    user_prompt=VAGUE_PROMPT,
    decision_engine=decision_engine,
    tool_client=tool_client,
    catalog=catalog,
    initial_layout=vague_layout,
    max_optimization_cycles=2,
    planner=RuleBasedPlanner(),
)

req = clarify_state.get("clarification_request")
print("Paused for clarification:", bool(req))
print("Buildings placed so far :", len(clarify_state.get("placed_buildings", [])))
print()
if req:
    print(req["summary"])
    for f in req["fields"]:
        tag = "REQUIRED" if f.get("critical") else "optional"
        print(f"  [{tag:8}] {f['key']:10} {f['question']}")
        print(f"             options: {f.get('options')}")
req

Paused for clarification: True
Buildings placed so far : 0

The prompt is a bit vague to place this accurately — could you confirm a few things?
  [REQUIRED] shape      Which building shape do you want?
             options: ['auto', 'I', 'L', 'T', 'U', 'H', 'Y', 'X', 'O']
  [REQUIRED] side       Which site side should the building sit next to?
             options: ['Main Street', 'East Service Street', 'Central Alignment Guide', 'north', 'south', 'east', 'west']
  [REQUIRED] view_side  Which side should the view be optimised toward?
             options: ['Main Street', 'East Service Street', 'Central Alignment Guide', 'north', 'south', 'east', 'west']
  [optional] size       Approximate footprint size per building?
             options: ['~600 m²', '~900 m²', '~1200 m²', '~1800 m²']
  [optional] use        What is the building use?
             options: ['residential', 'office', 'mixed']
  [optional] count      How many buildings?
             options: ['1', '2', '3', '4']


{'summary': 'The prompt is a bit vague to place this accurately — could you confirm a few things?',
 'fields': [{'key': 'shape',
   'question': 'Which building shape do you want?',
   'options': ['auto', 'I', 'L', 'T', 'U', 'H', 'Y', 'X', 'O'],
   'multi': False,
   'allow_custom': True,
   'critical': True},
  {'key': 'side',
   'question': 'Which site side should the building sit next to?',
   'options': ['Main Street',
    'East Service Street',
    'Central Alignment Guide',
    'north',
    'south',
    'east',
    'west'],
   'multi': False,
   'allow_custom': True,
   'critical': True},
  {'key': 'view_side',
   'question': 'Which side should the view be optimised toward?',
   'options': ['Main Street',
    'East Service Street',
    'Central Alignment Guide',
    'north',
    'south',
    'east',
    'west'],
   'multi': True,
   'allow_custom': True,
   'critical': True},
  {'key': 'size',
   'question': 'Approximate footprint size per building?',
   'options': ['~600 m²', '~9

In [15]:
# --- Answer the question (simulating the user clicking chips in ClarifyPanel) ---
answers = {
    "shape": "L",
    "side": "Main Street",      # frontage along the south edge
    "view_side": ["south"],
    "size": "~900 m²",
    "use": "office",
    "count": "1",
}

patched_brief, patched_layout = apply_clarification_answers(
    clarify_state.get("design_brief", {}),
    json.loads(clarify_state.get("layout_json", "{}")),
    answers,
    SITE_BOUNDARY,
)
# Resume: seed the answered brief + mark resolved so the agent proceeds (does not re-ask).
patched_layout["design_brief"] = patched_brief
patched_layout["clarification_resolved"] = True
patched_layout["interactive_clarification"] = True

# --- Run 2: the agent now has what it needs and places accurately ---
resolved_state = run_agent(
    user_prompt=VAGUE_PROMPT,
    decision_engine=decision_engine,
    tool_client=tool_client,
    catalog=catalog,
    initial_layout=patched_layout,
    max_optimization_cycles=3,
    planner=RuleBasedPlanner(),
)

print("Re-asked?           :", bool(resolved_state.get("clarification_request")))
print("Buildings placed    :", len(resolved_state.get("placed_buildings", [])))
print("Shape(s) used        :", [b.get("shape_preference") for b in resolved_state.get("design_brief", {}).get("buildings", [])])
print("Requested positions  :", json.loads(resolved_state.get("layout_json", "{}")).get("requested_positions"))
print("View target side(s)  :", json.loads(resolved_state.get("layout_json", "{}")).get("view_target_sides"))
print("Final response       :", (resolved_state.get("final_response") or "")[:200])

Re-asked?           : False
Buildings placed    : 1
Shape(s) used        : ['L']
Requested positions  : None
View target side(s)  : None
Final response       : Final design report (current best state)

Chosen geometry
- Building count: 1
- Shape: L-geometry, footprint area 900.0 sqm
- Wings: vertical wing ~670.5 sqm; horizontal wing ~454.5 sqm
- Selected pla


In [16]:
# Visualize the post-clarification placement
try:
    _clarified_shape = extract_latest_shape(resolved_state)
except Exception:
    _clarified_shape = None

make_site_plan_figure(
    SITE_BOUNDARY,
    current_shape=_clarified_shape,
    placed_buildings=resolved_state.get("placed_buildings", []),
    site_objects=SITE_OBJECTS,
    title="After clarification: building placed toward Main Street (answered: L-shape, south, office)",
)

## Tool surface check — view analysis & optimizers end-to-end

Directly exercises every geometry tool the agent (and the `/tools` API) uses — the same registry
behind `backend/routers/tools.py` — on this notebook's site, **independent of the LLM**. It confirms
view analysis (2D / 3D), attractor views, the NSGA-II single- and two-building optimizers, placement
sampling/ranking, objectives, and setbacks all import and run end-to-end. Pure-Python and
deterministic (no LLM / MCP), so it is a fast smoke test of the whole geometry layer.

In [17]:
import time

from agent.tools.view_analysis import evaluate_building_views, evaluate_attractor_views
from agent.tools.view_3d import evaluate_building_views_3d
from agent.tools.view_optimizer import (
    optimize_view_placement, optimize_two_building_placement,
    sample_valid_placements, rank_placements_by_view, list_objectives,
)
from agent.tools.site_setback import compute_buildable_zone, setback_summary

# Two sample footprints inside the 90x60 site, plus Main Street as a view attractor.
B = [[30, 20, 0], [60, 20, 0], [60, 40, 0], [30, 40, 0], [30, 20, 0]]
B2 = [[10, 42, 0], [34, 42, 0], [34, 56, 0], [10, 56, 0], [10, 42, 0]]
ATTR = [{"type": "line", "geometry": [[0, 8, 0], [90, 8, 0]]}]

checks = []
def check(name, fn):
    t = time.time()
    try:
        checks.append((name, "OK", f"{fn()}  ({time.time() - t:.2f}s)"))
    except Exception as exc:
        checks.append((name, "FAIL", f"{type(exc).__name__}: {exc}"))

sample = []
def _do_sample():
    global sample
    sample = sample_valid_placements(B, SITE_BOUNDARY)
    return f"{len(sample)} valid placements"

check("site_setback.setback_summary",        lambda: f"{len(setback_summary(SITE_BOUNDARY))} edges")
check("site_setback.compute_buildable_zone", lambda: f"buildable area={compute_buildable_zone(SITE_BOUNDARY).area:.0f} m²")
check("view_analysis.evaluate_building_views (2D)", lambda: f"view_score={evaluate_building_views(B, [], return_ray_detail=False)['view_score']:.3f}")
check("view_analysis.evaluate_attractor_views",     lambda: f"attractor_score={evaluate_attractor_views(B, ATTR, [], return_ray_detail=False)['attractor_score']:.3f}")
check("view_3d.evaluate_building_views_3d",         lambda: f"view_score_3d={evaluate_building_views_3d(B, 12.0, [], return_ray_detail=False)['view_score_3d']:.3f}")
check("view_optimizer.sample_valid_placements",     _do_sample)
check("view_optimizer.rank_placements_by_view",     lambda: f"top view_score={rank_placements_by_view(sample[:25], [])[0]['view_score']:.3f}")
check("view_optimizer.list_objectives",             lambda: f"{list_objectives()}")
check("view_optimizer.optimize_view_placement (NSGA-II)",
      lambda: f"{len(optimize_view_placement(boundary=B, site_boundary=SITE_BOUNDARY, obstacles=[], attractors=ATTR, population_size=24, generation_count=40)['pareto_solutions'])} pareto options")
check("view_optimizer.optimize_two_building_placement",
      lambda: f"{len(optimize_two_building_placement(boundary_1=B, boundary_2=B2, site_boundary=SITE_BOUNDARY, external_obstacles=[], min_building_separation=4.0, population_size=24, generation_count=50)['pareto_solutions'])} pareto options")

print(f"{'TOOL':52} {'STATUS':6} DETAIL")
print("-" * 100)
for name, status, detail in checks:
    print(f"{name:52} {status:6} {detail}")
n_ok = sum(1 for _, s, _ in checks if s == "OK")
print(f"\n{n_ok}/{len(checks)} tools OK")
assert n_ok == len(checks), "Some tools failed — see the FAIL rows above."

TOOL                                                 STATUS DETAIL
----------------------------------------------------------------------------------------------------
site_setback.setback_summary                         OK     6 edges  (0.00s)
site_setback.compute_buildable_zone                  OK     buildable area=4000 m²  (0.00s)
view_analysis.evaluate_building_views (2D)           OK     view_score=1.000  (0.00s)
view_analysis.evaluate_attractor_views               OK     attractor_score=1.000  (0.00s)
view_3d.evaluate_building_views_3d                   OK     view_score_3d=1.000  (0.00s)
view_optimizer.sample_valid_placements               OK     506 valid placements  (0.25s)
view_optimizer.rank_placements_by_view               OK     top view_score=1.000  (0.01s)
view_optimizer.list_objectives                       OK     ['attractor_view', 'clearance_from_obstacles', 'clearance_from_site', 'sky_exposure', 'unblocked_view']  (0.00s)
view_optimizer.optimize_view_placement (NSGA